# Session 1: OpenAI API Engineering with Structured Outputs (Instructor Notebook -- Full Solutions)

This is the **instructor version** of Session 1. It contains the same demos as the student notebook, plus **complete, working solutions** for all four tasks. Use this as your reference during class delivery.

## Learning Objectives

By the end of this session, students will be able to:

1. **Use streaming** and track token usage for cost management
2. **Generate structured JSON** outputs using `response_format` parameter
3. **Define and use function calling** (tool use) with the OpenAI API
4. **Validate LLM outputs** with Pydantic models for type safety
5. **Build extraction pipelines** that turn unstructured text into structured data
6. **Implement robust API wrappers** with retries and error handling

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import os
import time
from pydantic import BaseModel, Field
from typing import Optional, List

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: OpenAI API Deep Dive — Streaming, Token Usage, and Finish Reasons

In production agentic systems you need to: (a) track token usage for cost management, (b) stream responses for real-time UX, and (c) inspect finish reasons to know if the model completed its response or was cut off.

In [ ]:
# Demo 1 - OpenAI API Deep Dive

client = openai.OpenAI()

# Part A: Token usage tracking
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain quantum computing in 3 sentences."}],
    max_tokens=150
)

print("=== Token Usage ===")
print(f"Prompt tokens    : {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens     : {response.usage.total_tokens}")
print(f"Finish reason    : {response.choices[0].finish_reason}")
print(f"\nResponse:\n{response.choices[0].message.content}")

# Part B: Streaming responses
print("\n=== Streaming Response ===")
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Count from 1 to 5, one number per line."}],
    max_tokens=50,
    stream=True
)

collected_text = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        collected_text += token
        print(token, end="", flush=True)

print(f"\n\nFull collected text: {collected_text}")

### Demo 2: Structured Output with JSON Mode

By setting `response_format={"type": "json_object"}`, the model is guaranteed to return valid JSON. You **must** include the word "JSON" in your prompt when using this mode.

In [ ]:
# Demo 2 - Structured Output with JSON Mode

client = openai.OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that outputs JSON."},
        {"role": "user", "content": "Give me information about Python as a programming language. Include: name, year_created, creator, paradigms (list), and a short description."}
    ],
    response_format={"type": "json_object"},
    max_tokens=300
)

raw_json = response.choices[0].message.content
print("Raw JSON response:")
print(raw_json)

parsed = json.loads(raw_json)
print("\nParsed and formatted:")
print(json.dumps(parsed, indent=2))

print(f"\nLanguage: {parsed.get('name', 'N/A')}")
print(f"Created : {parsed.get('year_created', 'N/A')}")
print(f"Creator : {parsed.get('creator', 'N/A')}")

### Demo 3: Function Calling — Defining and Using Tools

Function calling lets the model decide **when** and **how** to call external tools. The flow is:
1. Define tool schemas and send them with the request
2. Model returns a `tool_calls` response
3. Execute the function locally
4. Send the result back for a final answer

In [ ]:
# Demo 3 - Function Calling / Tool Use

client = openai.OpenAI()

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Temperature unit"}
                },
                "required": ["city"]
            }
        }
    }
]

def get_weather(city, unit="celsius"):
    weather_data = {
        "San Francisco": {"temp": 18, "condition": "Foggy"},
        "New York": {"temp": 25, "condition": "Sunny"},
        "London": {"temp": 15, "condition": "Rainy"},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown"})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9/5 + 32)
    return json.dumps({"city": city, "temperature": data["temp"], "unit": unit, "condition": data["condition"]})

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What's the weather like in San Francisco?"}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")

if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"Function called: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")
    args = json.loads(tool_call.function.arguments)
    result = get_weather(**args)
    print(f"Function result: {result}")

    follow_up = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "What's the weather like in San Francisco?"},
            message,
            {"role": "tool", "tool_call_id": tool_call.id, "content": result}
        ],
        tools=tools
    )
    print(f"\nFinal response:\n{follow_up.choices[0].message.content}")
else:
    print(f"Direct response: {message.content}")

### Demo 4: Pydantic-Based Response Validation

Pydantic models let you define the **exact shape** of the data you expect, with type checking and constraints.

In [ ]:
# Demo 4 - Pydantic-Based Response Validation

client = openai.OpenAI()

class MovieReview(BaseModel):
    title: str = Field(description="The movie title")
    year: int = Field(description="Release year")
    rating: float = Field(ge=0, le=10, description="Rating out of 10")
    genre: str = Field(description="Primary genre")
    summary: str = Field(description="One-sentence summary")
    recommended: bool = Field(description="Whether you recommend this movie")

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a movie critic. Output your review as JSON with these exact fields: title (string), year (integer), rating (float 0-10), genre (string), summary (string), recommended (boolean)."},
        {"role": "user", "content": "Review the movie 'Inception' by Christopher Nolan."}
    ],
    response_format={"type": "json_object"},
    max_tokens=200
)

raw = response.choices[0].message.content
print("Raw JSON:")
print(raw)

try:
    review = MovieReview.model_validate_json(raw)
    print("\nValidated MovieReview:")
    print(f"  Title      : {review.title}")
    print(f"  Year       : {review.year}")
    print(f"  Rating     : {review.rating}/10")
    print(f"  Genre      : {review.genre}")
    print(f"  Summary    : {review.summary}")
    print(f"  Recommended: {'Yes' if review.recommended else 'No'}")
except Exception as e:
    print(f"\nValidation error: {e}")

### Demo 5: Building a Structured Data Extraction Pipeline

This demo combines JSON mode and Pydantic into a reusable pipeline that extracts structured contact information from unstructured text.

In [ ]:
# Demo 5 - Building a Structured Data Extraction Pipeline

client = openai.OpenAI()

class ContactInfo(BaseModel):
    name: str = Field(description="Full name of the person")
    email: Optional[str] = Field(default=None, description="Email address if found")
    phone: Optional[str] = Field(default=None, description="Phone number if found")
    company: Optional[str] = Field(default=None, description="Company name if found")
    role: Optional[str] = Field(default=None, description="Job title or role if found")

def extract_contacts(text: str) -> List[ContactInfo]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract all contact information from the text. Return a JSON object with a 'contacts' key containing a list. Each contact should have: name, email (null if not found), phone (null if not found), company (null if not found), role (null if not found)."},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        max_tokens=500
    )
    data = json.loads(response.choices[0].message.content)
    contacts = []
    for item in data.get("contacts", []):
        try:
            contacts.append(ContactInfo(**item))
        except Exception as e:
            print(f"Skipping invalid contact: {e}")
    return contacts

sample_text = """Hi, I'm Sarah Chen from TechCorp. You can reach me at sarah.chen@techcorp.com
or call (555) 123-4567. I'm the VP of Engineering.

Also, please loop in our CTO, James Rodriguez. His email is james.r@techcorp.com.

For billing questions, contact accounts@techcorp.com -- ask for Maria Lopez."""

print("Extracted contacts:")
print("=" * 60)
contacts = extract_contacts(sample_text)
for i, contact in enumerate(contacts, 1):
    print(f"\nContact {i}:")
    print(f"  Name   : {contact.name}")
    print(f"  Email  : {contact.email or 'N/A'}")
    print(f"  Phone  : {contact.phone or 'N/A'}")
    print(f"  Company: {contact.company or 'N/A'}")
    print(f"  Role   : {contact.role or 'N/A'}")

---
## Tasks -- Full Solutions

Below are the complete, working solutions for all four student tasks. Each solution is preceded by an explanation of the approach.

### Task 1: Build a Structured Entity Extractor Using JSON Mode

**Approach:** The solution uses JSON mode with a system message that defines the expected schema (people, organizations, locations, dates). We parse the JSON response and return it as a dictionary. The key teaching point is how the system message acts as a schema definition.

In [ ]:
# Task 1 - SOLUTION: Build a Structured Entity Extractor Using JSON Mode

def extract_entities(article_text):
    """Extract named entities from a news article using JSON mode."""
    client = openai.OpenAI()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "Extract named entities from the text. Return a JSON object with these keys: "
                "people (list of strings), organizations (list of strings), "
                "locations (list of strings), dates (list of strings). "
                "If a category has no entities, use an empty list."
            )},
            {"role": "user", "content": article_text}
        ],
        response_format={"type": "json_object"},
        max_tokens=300
    )

    return json.loads(response.choices[0].message.content)


# Test
sample_article = (
    "Apple CEO Tim Cook announced at WWDC in San Jose on June 10, 2024 that "
    "the company would partner with OpenAI to integrate ChatGPT into iOS. "
    "Microsoft CEO Satya Nadella responded from Redmond, praising the collaboration."
)
entities = extract_entities(sample_article)
print("Extracted entities:")
print(json.dumps(entities, indent=2))

### Task 2: Implement a Calculator Tool with Function Calling

**Approach:** We define a `calculate` tool with an operation enum and two number parameters. The `ask_math_question` function handles the full loop: initial request, detecting the tool call, executing the function, and sending the result back for a natural-language answer.

In [ ]:
# Task 2 - SOLUTION: Implement a Calculator Tool with Function Calling

def calculate(operation, a, b):
    """Perform a math operation on two numbers."""
    operations = {
        "add": lambda x, y: x + y,
        "subtract": lambda x, y: x - y,
        "multiply": lambda x, y: x * y,
        "divide": lambda x, y: x / y if y != 0 else "Error: division by zero"
    }
    if operation not in operations:
        return f"Error: unknown operation '{operation}'"
    return operations[operation](a, b)


def ask_math_question(question):
    """Send a math question and handle function calling."""
    client = openai.OpenAI()

    calc_tool = [{
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Perform a mathematical operation on two numbers",
            "parameters": {
                "type": "object",
                "properties": {
                    "operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]},
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"}
                },
                "required": ["operation", "a", "b"]
            }
        }
    }]

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        tools=calc_tool,
        tool_choice="auto"
    )

    message = response.choices[0].message

    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        result = calculate(**args)

        follow_up = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "user", "content": question},
                message,
                {"role": "tool", "tool_call_id": tool_call.id, "content": str(result)}
            ],
            tools=calc_tool
        )
        return follow_up.choices[0].message.content

    return message.content


# Test
print(ask_math_question("What is 42 multiplied by 17?"))
print()
print(ask_math_question("Divide 144 by 12"))

### Task 3: Create a Multi-Tool Agent with Automatic Tool Dispatch

**Approach:** We define three tools (weather, calculator, time), implement each as a simulated function, and use a dispatch dictionary to route tool calls. The agent handles both single and multiple tool calls, plus direct responses when no tool is needed.

In [ ]:
# Task 3 - SOLUTION: Create a Multi-Tool Agent with Automatic Tool Dispatch

# Tool implementations
def tool_get_weather(city):
    weather_data = {
        "London": {"temp": 15, "condition": "Rainy"},
        "Tokyo": {"temp": 28, "condition": "Humid"},
        "Paris": {"temp": 22, "condition": "Sunny"}
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown"})
    return json.dumps({"city": city, "temperature": data["temp"], "condition": data["condition"]})

def tool_calculate(operation, a, b):
    ops = {"add": a + b, "subtract": a - b, "multiply": a * b, "divide": a / b if b != 0 else "Error"}
    return json.dumps({"operation": operation, "a": a, "b": b, "result": ops.get(operation, "Unknown")})

def tool_get_time(timezone):
    times = {"UTC": "12:00", "EST": "07:00", "PST": "04:00", "JST": "21:00", "GMT": "12:00"}
    return json.dumps({"timezone": timezone, "current_time": times.get(timezone, "12:00")})

# Tool definitions
tools = [
    {"type": "function", "function": {"name": "get_weather", "description": "Get weather for a city",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "calculate", "description": "Perform math on two numbers",
        "parameters": {"type": "object", "properties": {"operation": {"type": "string", "enum": ["add", "subtract", "multiply", "divide"]}, "a": {"type": "number"}, "b": {"type": "number"}}, "required": ["operation", "a", "b"]}}},
    {"type": "function", "function": {"name": "get_time", "description": "Get current time in a timezone",
        "parameters": {"type": "object", "properties": {"timezone": {"type": "string"}}, "required": ["timezone"]}}}
]

# Dispatch table
dispatch = {
    "get_weather": lambda args: tool_get_weather(args["city"]),
    "calculate": lambda args: tool_calculate(args["operation"], args["a"], args["b"]),
    "get_time": lambda args: tool_get_time(args["timezone"])
}

def multi_tool_agent(user_message):
    client = openai.OpenAI()
    messages = [{"role": "user", "content": user_message}]

    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=tools, tool_choice="auto"
    )
    message = response.choices[0].message

    if message.tool_calls:
        messages.append(message)
        for tc in message.tool_calls:
            args = json.loads(tc.function.arguments)
            result = dispatch[tc.function.name](args)
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        final = client.chat.completions.create(
            model="gpt-4o-mini", messages=messages, tools=tools
        )
        return final.choices[0].message.content

    return message.content


# Test
print(multi_tool_agent("What's the weather in London?"))
print()
print(multi_tool_agent("Calculate 25 times 4"))
print()
print(multi_tool_agent("What time is it in Tokyo?"))

### Task 4: Build a Robust API Client with Retries, Validation, and Streaming

**Approach:** The `RobustLLMClient` class wraps the OpenAI client with: (1) exponential backoff retry logic, (2) optional Pydantic validation via `model_validate_json()`, (3) streaming support that collects tokens in real-time, and (4) cumulative token usage tracking. This is a production-ready pattern.

In [ ]:
# Task 4 - SOLUTION: Build a Robust API Client with Retries, Validation, and Streaming

class RobustLLMClient:
    def __init__(self, model="gpt-4o-mini", max_retries=3):
        self.client = openai.OpenAI()
        self.model = model
        self.max_retries = max_retries
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_calls = 0

    def call(self, messages, response_model=None, stream=False, **kwargs):
        """Make a robust API call with retries and optional validation."""
        for attempt in range(self.max_retries):
            try:
                if stream:
                    return self._stream_call(messages, **kwargs)

                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    response_format={"type": "json_object"} if response_model else None,
                    **kwargs
                )

                self.total_prompt_tokens += response.usage.prompt_tokens
                self.total_completion_tokens += response.usage.completion_tokens
                self.total_calls += 1

                content = response.choices[0].message.content

                if response_model:
                    return response_model.model_validate_json(content)
                return content

            except Exception as e:
                wait = 2 ** attempt
                print(f"Attempt {attempt + 1} failed: {e}. Retrying in {wait}s...")
                if attempt < self.max_retries - 1:
                    time.sleep(wait)
                else:
                    raise

    def _stream_call(self, messages, **kwargs):
        stream = self.client.chat.completions.create(
            model=self.model, messages=messages, stream=True, **kwargs
        )
        collected = ""
        for chunk in stream:
            if chunk.choices[0].delta.content:
                token = chunk.choices[0].delta.content
                collected += token
                print(token, end="", flush=True)
        print()
        self.total_calls += 1
        return collected

    def get_usage_stats(self):
        return {
            "total_calls": self.total_calls,
            "total_prompt_tokens": self.total_prompt_tokens,
            "total_completion_tokens": self.total_completion_tokens,
            "total_tokens": self.total_prompt_tokens + self.total_completion_tokens
        }


# Test non-streaming with validation
client = RobustLLMClient()

class LanguageInfo(BaseModel):
    name: str
    year: int
    creator: str

result = client.call(
    messages=[
        {"role": "system", "content": "Output JSON with fields: name, year, creator"},
        {"role": "user", "content": "Tell me about Python"}
    ],
    response_model=LanguageInfo
)
print(f"Validated: {result.name}, created {result.year} by {result.creator}")
print(f"Usage: {client.get_usage_stats()}")

# Test streaming
print("\nStreaming test:")
client.call(
    messages=[{"role": "user", "content": "Count from 1 to 3 briefly."}],
    stream=True,
    max_tokens=50
)
print(f"Usage after streaming: {client.get_usage_stats()}")

---
## Summary

In this session students learned production-grade OpenAI API engineering skills:

1. **API Deep Dive** -- Streaming responses for real-time output, tracking token usage for cost management, and understanding finish reasons.
2. **JSON Mode** -- Using `response_format={"type": "json_object"}` to get reliable, parseable structured data from the model.
3. **Function Calling** -- Defining tools that the model can invoke, handling the request-execute-respond loop that powers agentic tool use.
4. **Pydantic Validation** -- Using Pydantic models to validate and parse LLM outputs with type safety and automatic error detection.
5. **Extraction Pipelines** -- Combining JSON mode and Pydantic into reusable pipelines that extract structured data from unstructured text.

**Instructor Notes:**
- Emphasize that structured outputs are the bridge between LLMs and code -- without them, agents cannot act on LLM decisions.
- For Task 2, discuss how the model sometimes tries to answer math questions directly -- show that `tool_choice="required"` forces tool use.
- For Task 4, discuss production concerns: rate limits, cost tracking, and graceful degradation.

**Up Next:** In Session 2, we will move from raw API calls to LangChain, learning how to compose modular chains, integrate tools, and build retrieval-augmented generation (RAG) systems.